# 🔢 Digit Detective — Track A starter

**Run all cells from top to bottom in Google Colab.** Uses scikit-learn's built-in 8×8 digits dataset; no API key, paid service, or separate download.

A ready-to-run baseline **does not count as an original competition entry**. Change something meaningful, rerun evaluation, and document how your changes performed.

Ideas: visualize mistakes; compare classifiers using the same test split; build a small user interface; explain where the model makes errors.

In [ ]:
"""Reproducible handwritten digit classification starter.

Uses the built-in scikit-learn digits dataset: no downloads or API credentials.
Participants should change and measure the baseline, not submit it unchanged.
"""
from __future__ import annotations

import numpy as np
from sklearn.datasets import load_digits
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


def train_digit_model(random_state: int = 42) -> dict:
    """Train on 75% of examples and evaluate once on untouched 25% test data."""
    digits = load_digits()
    x_train, x_test, y_train, y_test = train_test_split(
        digits.data, digits.target, test_size=0.25,
        random_state=random_state, stratify=digits.target,
    )
    model = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=2000, random_state=random_state),
    )
    model.fit(x_train, y_train)
    predictions = model.predict(x_test)
    return {
        "model": model,
        "x_train": x_train, "x_test": x_test,
        "y_train": y_train, "y_test": y_test,
        "predictions": predictions,
        "accuracy": float(accuracy_score(y_test, predictions)),
        "report": classification_report(y_test, predictions, zero_division=0),
        "confusion_matrix": confusion_matrix(y_test, predictions),
    }


def predict_digit(model, pixels) -> dict:
    """Predict a digit from exactly 64 grayscale feature values (8x8 image)."""
    array = np.asarray(pixels, dtype=float)
    if array.size != 64 or not np.all(np.isfinite(array)):
        raise ValueError("Expected exactly 64 finite grayscale values.")
    if np.any(array < 0) or np.any(array > 16):
        raise ValueError("Built-in digits images use values from 0 to 16.")
    probabilities = model.predict_proba(array.reshape(1, 64))[0]
    best = int(np.argmax(probabilities))
    return {
        "digit": int(model.classes_[best]),
        "confidence": float(probabilities[best]),
        "probabilities": probabilities.tolist(),
    }


In [ ]:
from matplotlib import pyplot as plt
baseline = train_digit_model()
print("Held-out accuracy:", round(baseline["accuracy"], 3))
print(baseline["report"])
print("Confusion matrix:\n", baseline["confusion_matrix"])

fig, axes = plt.subplots(1, 5, figsize=(12, 3))
for index, axis in enumerate(axes):
    image = baseline["x_test"][index].reshape(8, 8)
    result = predict_digit(baseline["model"], baseline["x_test"][index])
    axis.imshow(image, cmap="gray_r")
    axis.set_title(f"Actual {baseline['y_test'][index]}\nPred {result['digit']}")
    axis.axis("off")
plt.tight_layout()
plt.show()

## Your team's experiment

1. Write down the baseline accuracy above.
2. Change one meaningful aspect of the classifier or its interface.
3. Evaluate fairly on the **same untouched test set**. Do not tune repeatedly on test labels; use a validation split or cross-validation for experiments.
4. Record what changed, what you expected, the actual outcome, and one limitation.
5. Publish your *own* repository with a README and public license, and submit using the official event instructions.

**Safety:** This is an educational classifier, not a handwriting identity system.

In [ ]:
# Team experiment: replace this comment with YOUR extension.
# For example, find and display the first three misclassified test images.
errors = [i for i, (truth, predicted) in enumerate(zip(baseline["y_test"], baseline["predictions"])) if truth != predicted]
print("Misclassified test samples:", len(errors))
print("First few indices:", errors[:3])